## script to wrangle Xenium data: parse cell_feature_matrix.h5 file to generate
1. Gene expression

An optional retangle qupath annotation file can be provided to focus on cells in the rectangle, needs
- geojson file
- cells.zarr.zip file

In [85]:
import sys, os
sys.path.insert(0, "/home/ubuntu/xenaConvert/")
import zarr
from xenaConvert import *
import scanpy as sc

In [108]:
xenium_bundle_dir = "/mnt/windows/16-080L/Xenium/output-XETG00049__0015980__XE009-80-C__20231220__211433/"
max_fraction = 0.2 # for xenium data, default in scanpy is 0.05
count_file = "cell_feature_matrix.h5"
count_file = os.path.join(xenium_bundle_dir, count_file)
h5ad_file = "adata.h5ad" # optional for saving data to disk
outputdir =  "xena"
study = "16-080L xenium"
selected_cells = None

## (optional) Select a subset of cells by a rectangle geojsonfile (from Qupath single rectangle annotation)
**selected_cells**

In [90]:
geojsonfile = "imgs/morphology_16-080L.geojson"
cell_zarr_file = "cells.zarr.zip"
cell_zarr_file = os.path.join(xenium_bundle_dir, cell_zarr_file)
px_to_um = 0.2125

In [91]:
J = json.load(open(geojsonfile, 'r'))
rectangle = pd.DataFrame(J["features"][0]["geometry"]["coordinates"][0])
# x
x_min = min(rectangle[0]) * px_to_um
x_max = max(rectangle[0]) * px_to_um
# y
y_min = min(rectangle[1]) * px_to_um
y_max = max(rectangle[1]) * px_to_um

In [92]:
root = zarr.open(cell_zarr_file, mode='r')

In [93]:
def convert_to_xenium_cell_id (args):
    cell_id_prefix, dataset_suffix = args
    # print (cell_id_prefix, hex(cell_id_prefix)[2:])
    #  [0 - 9, a - f] to the range a - p
    hex_to_xenium = {
        "0":"a",
        "1":"b",
        "2":"c",
        "3":"d",
        "4":"e",
        "5":"f",
        "6":"g",
        "7":"h",
        "8":"i",
        "9":"j",
        "a":"k",
        "b":"l",
        "c":"m",
        "d":"n",
        "e":"o",
        "f":"p"
    }
    def convert_hex_to_xenium(hex):
        return hex_to_xenium[hex]
        
    cell_id_prefix_xenium = "".join(list(map(convert_hex_to_xenium, hex(cell_id_prefix)[2:])))
    cell_id_prefix_xenium = "a" * (8 - len(cell_id_prefix_xenium)) + cell_id_prefix_xenium
    cell_id =  cell_id_prefix_xenium + "-" + str(dataset_suffix)
    return cell_id

In [94]:
xenium_cell_id = list(map(convert_to_xenium_cell_id, list(zip(root["cell_id"][:,0], root["cell_id"][:,1]))))

In [95]:
df = pd.DataFrame(root["cell_summary"], 
                  columns = ["cell_centroid_x", "cell_centroid_y", "cell_area", "nucleus_centroid_x","nucleus_centroid_y", "nucleus_area", "z_level"], 
                  index = xenium_cell_id)
df

,cell_centroid_x,cell_centroid_y,cell_area,nucleus_centroid_x,nucleus_centroid_y,nucleus_area,z_level
aaaaadfk-1,822.971619,7299.142578,353.708919,826.048523,7305.054688,28.854845,21.0
aaaabcck-1,828.437805,7333.003418,217.924070,827.635437,7335.468750,4.605938,24.0
aaaaidhj-1,840.336853,7343.100586,100.879066,841.786987,7344.538086,16.888438,21.0
aaaaiheo-1,809.179138,7597.605469,158.137193,809.072632,7594.875488,14.540313,21.0
aaaaokan-1,832.967529,7317.130371,55.948596,833.934509,7313.253906,5.734844,18.0
...,...,...,...,...,...,...,...
oijcibae-1,7508.504395,886.216248,143.867818,7509.102539,886.409424,38.789220,24.0
oijcjhcf-1,7516.332031,883.460083,142.784068,7513.018066,882.607910,34.363907,21.0
oijddlfb-1,7511.994141,875.118286,68.682659,7513.358398,874.333252,14.901563,18.0
oijdemdk-1,7770.033691,10998.489258,1001.023786,7770.152344,10998.171875,28.538751,30.0


In [96]:
if geojsonfile:
    selected_df = df[(df.cell_centroid_x> x_min) 
                & (df.cell_centroid_x < x_max) 
                & (df.cell_centroid_y > y_min)
                & (df.cell_centroid_y < y_max)
    ]
    selected_cells = list(selected_df.index)
    len(selected_cells)

In [97]:
len(selected_cells)

27521

# Wrange .h5 gene expression data

## for analysis

In [98]:
adata = sc.read_10x_h5(count_file)

In [99]:
if selected_cells:
    adata = adata[selected_cells]

In [100]:
adata

View of AnnData object with n_obs × n_vars = 27521 × 377
    var: 'gene_ids', 'feature_types', 'genome'

In [101]:
adata.X[:,:]

<27521x377 sparse matrix of type '<class 'numpy.float32'>'
	with 549079 stored elements in Compressed Sparse Row format>

In [102]:
adata = basic_analysis(adata, max_fraction = max_fraction)

/home/ubuntu/singlecellenv/lib/python3.10/site-packages/scanpy/preprocessing/_simple.py:140: ImplicitModificationWarning: Trying to modify attribute `.obs` of view, initializing view as actual.
  adata.obs['n_genes'] = number
/home/ubuntu/singlecellenv/lib/python3.10/site-packages/scanpy/preprocessing/_normalization.py:197: UserWarning: Some cells have zero counts
  warn(UserWarning('Some cells have zero counts'))


In [109]:
adataToCluster(adata, outputdir, study)

In [110]:
adataToMap(adata, outputdir, study)

unrecognized or ignored map: X_pca


## for generate all xena data files

In [34]:
def tenxh5ToXena (count_file, selected_cells = None):
    adata = sc.read_10x_h5(count_file)
    if selected_cells:
        adata = adata[selected_cells]
    rawX = adata.X
    adata = basic_analysis(adata)
    adataToXena(adata, outputdir, study, rawX = rawX)
    return adata

In [ ]:
adata = tenxh5ToXena (count_file, selected_cells)

In [48]:
# to save result to disk (optional)
adata.write_h5ad(h5ad_file)

In [4]:
# read back from save h5ad file
adata = sc.read_h5ad(h5ad_file)